# Introduction

This work presents a constrained combinatorial optimization approach to the **Sports League Assignment Problem** using **Genetic Algorithms (GAs)**. The objective is to allocate a fixed pool of professional players into a set of 5 structurally valid teams in such a way that the **standard deviation of the teams\" average skill ratings** is minimized—promoting competitive balance across the league.

Each player is defined by three attributes: **position** (one of `GK`, `DEF`, `MID`, `FWD`), **skill rating** (a numerical measure of ability), and **cost** (in million euros). A valid solution must satisfy the following **hard constraints**:

- Each team must consist of exactly **7 players**, with a specific positional structure: **1 GK, 2 DEF, 2 MID, and 2 FWD**
- Each team must have a **total cost ≤ 750 million €**
- Each player must be assigned to **exactly one team** (no overlaps)

The **search space** is therefore highly constrained and discrete, and infeasible configurations are explicitly excluded from the solution space. The optimization objective is to identify league configurations where teams are not only valid but also **skill-balanced**, quantified by the **standard deviation of average skill ratings across teams**, which serves as the **fitness function** (to be minimized).

To address this, we implement a domain-adapted **Genetic Algorithm framework** featuring:

- A custom **representation** based on team-to-player mappings
- Validity-preserving **mutation** and **crossover** operators
- Multiple **selection mechanisms**
- Optional **elitism** and population-level diversity handling

This report provides a formal problem definition, details the design of the solution encoding and operators, and presents empirical results comparing different GA configurations. The overall objective is to evaluate how well GA-based metaheuristics can navigate this complex constrained search space and evolve solutions that both satisfy domain constraints and optimize league balance.

In addition to Genetic Algorithms, this project also explores and evaluates alternative optimization strategies, such as **Hill Climbing** and **Simulated Annealing**, which are well-suited for navigating discrete and constrained search spaces. These algorithms offer different trade-offs in terms of exploration, exploitation, and convergence speed. By implementing and benchmarking multiple approaches on the same problem, we aim to gain deeper insights into their relative effectiveness and robustness when applied to complex constrained optimization tasks such as the Sports League Assignment. This comparative analysis enhances the interpretability of results and supports a broader understanding of the strengths and limitations of population-based versus local search-based heuristics.

# Load libraries and data import

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os 

from solution import LeagueSolution, LeagueHillClimbingSolution, LeagueSASolution
from evolution import genetic_algorithm, hill_climbing, simulated_annealing 
from operators import (
    # Base Mutations
    mutate_swap,
    mutate_team_shift,
    mutate_shuffle_team,
    # New/Adapted Mutations
    mutate_swap_constrained,
    mutate_targeted_player_exchange,
    mutate_shuffle_within_team_constrained,
    # Base Crossovers
    crossover_one_point,
    crossover_uniform,
    # New/Adapted Crossovers
    crossover_one_point_prefer_valid,
    crossover_uniform_prefer_valid,
    # Selection Operators
    selection_ranking,
    selection_tournament_variable_k,
    selection_boltzmann
)

# Define the directory for saving graphs and results

IMAGES_SP_DIR = os.path.join('..', 'images_sp')
# Ensure the directory exists
if not os.path.exists(IMAGES_SP_DIR):
    os.makedirs(IMAGES_SP_DIR)
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Created directory: {IMAGES_SP_DIR}")

# Load player data
players_df = pd.read_csv("players.csv", sep=";")
players_data = players_df.to_dict(orient="records") # Renamed to players_data for clarity

# Data Inspection, and Experiment Parameters

In [20]:
all_results_summary = []

# Define problem parameters
NUM_TEAMS = 5
TEAM_SIZE = 7
MAX_BUDGET = 750

# Parameter for number of runs (e.g., 10, 30) - TEST RUN
NUM_RUNS = 1

## Problem Representation

Player-assignment Representation (Linear Encoding) 

Let:

- $P = \{p_1, p_2, \dots, p_{35}\}$ be the set of players  
- $T = \{0, 1, 2, 3, 4\}$ be team IDs

A solution is represented by a vector:

$$
\mathbf{a} = [a_1, a_2, \dots, a_{35}] \in T^{35}
$$

where $a_i$ is the team assignment for player $p_i$.

**Team Definitions:**

$$
P_j = \{p_i \in P \mid a_i = j\}, \quad \forall j \in T
$$

**Constraints:**

$$
|P_j| = 7 \quad \text{and}
$$

$$
\begin{aligned}
&|\{p \in P_j \mid pos(p) = \text{GK}\}| = 1 \\
&|\{p \in P_j \mid pos(p) = \text{DEF}\}| = 2 \\
&|\{p \in P_j \mid pos(p) = \text{MID}\}| = 2 \\
&|\{p \in P_j \mid pos(p) = \text{FWD}\}| = 2 \\
&\sum_{p \in P_j} cost(p) \leq 750
\end{aligned}
$$

**Objective Function:**

$$
f(\mathbf{a}) = \sigma\left(\left\{\frac{1}{7} \sum_{p \in P_j} skill(p) \,\middle|\, j \in T\right\}\right)
$$

# Experiment configuration

In [ ]:
def get_experiment_config(custom_config=None):
    """
    Define default experiment configuration parameters and override with any custom settings.
    
    Args:
        custom_config (dict): Optional dictionary with custom parameter values to override defaults
        
    Returns:
        dict: Complete configuration dictionary
    """
    # Default configuration
    config = {
        # Problem parameters
        "num_teams": 5,
        "team_size": 7,
        "max_budget": 750,
        
        # Experiment parameters
        "num_runs": 1,
        "images_dir": IMAGES_SP_DIR,  # Assuming this is defined elsewhere
        "random_seed": 42,
        
        # Hill Climbing parameters
        "hc_configs": [
            {"name": "HC_Default", "max_iterations": 1000},
            {"name": "HC_LongRun", "max_iterations": 2000},
            {"name": "HC_ShortRun", "max_iterations": 500}
        ],
        
        # Simulated Annealing parameters
        "sa_configs": [
            {"name": "SA_Default", "initial_temp": 1000, "final_temp": 1, "alpha": 0.99, "iterations_per_temp": 100},
            {"name": "SA_MoreExploration", "initial_temp": 5000, "final_temp": 0.1, "alpha": 0.995, "iterations_per_temp": 200},
            {"name": "SA_FasterConvergence", "initial_temp": 1000, "final_temp": 0.1, "alpha": 0.95, "iterations_per_temp": 150},
            {"name": "SA_Balanced", "initial_temp": 2000, "final_temp": 0.5, "alpha": 0.98, "iterations_per_temp": 150},
        ],
        
        # Genetic Algorithm parameters
        "ga_configs": [
            {"name": "GA_Config_1_SwapConst1PtPreferVTournVarK", "pop_size": 50, "gens": 75, "mut_rate": 0.2, "elite": 5, 
             "mut_op": "mutate_swap_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_tournament_variable_k", "tourn_k": 3, "boltz_temp": None},
            
            {"name": "GA_Config_2_TargetExchUnifPreferVRank", "pop_size": 50, "gens": 75, "mut_rate": 0.2, "elite": 5, 
             "mut_op": "mutate_targeted_player_exchange", "cross_op": "crossover_uniform_prefer_valid", 
             "sel_op": "selection_ranking", "tourn_k": None, "boltz_temp": None},
            
            {"name": "GA_Config_3_ShuffleTeamConst1PtPreferVBoltz", "pop_size": 50, "gens": 75, "mut_rate": 0.2, "elite": 5, 
             "mut_op": "mutate_shuffle_within_team_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_boltzmann", "tourn_k": None, "boltz_temp": 100},
            
            {"name": "GA_Config_4_SwapConst1PtPreferVTournVarK_150Gen", "pop_size": 50, "gens": 150, "mut_rate": 0.2, "elite": 5, 
             "mut_op": "mutate_swap_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_tournament_variable_k", "tourn_k": 3, "boltz_temp": None},
            
            {"name": "GA_Config_5_SwapConst1PtPreferVTournVarK_100Pop", "pop_size": 100, "gens": 75, "mut_rate": 0.2, "elite": 10, 
             "mut_op": "mutate_swap_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_tournament_variable_k", "tourn_k": 3, "boltz_temp": None},
            
            {"name": "GA_Config_6_SwapConst1PtPreferVTournVarK_MutRate0.1", "pop_size": 50, "gens": 75, "mut_rate": 0.1, "elite": 5, 
             "mut_op": "mutate_swap_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_tournament_variable_k", "tourn_k": 3, "boltz_temp": None},
            
            {"name": "GA_Config_7_SwapConst1PtPreferVTournVarK_MutRate0.3", "pop_size": 50, "gens": 75, "mut_rate": 0.3, "elite": 5, 
             "mut_op": "mutate_swap_constrained", "cross_op": "crossover_one_point_prefer_valid", 
             "sel_op": "selection_tournament_variable_k", "tourn_k": 3, "boltz_temp": None},
        ],
        
        # Plotting parameters
        "figure_size": (12, 7),
        "dpi": 100,
        "line_width": 2,
        
        # Run control - which algorithm sections to run
        "run_sections": {
            "hill_climbing": True,
            "simulated_annealing": True, 
            "genetic_algorithm": True,
            "hc_comparison": True,
            "sa_comparison": True,
            "ga_comparison": True,
            "all_comparison": True
        }
    }
    
    # Override defaults with any custom settings
    if custom_config:
        for key, value in custom_config.items():
            config[key] = value
    
    # For GA configs, convert string operator names to actual function references
    from operators import (
        mutate_swap_constrained,
        mutate_targeted_player_exchange,
        mutate_shuffle_within_team_constrained,
        crossover_one_point_prefer_valid,
        crossover_uniform_prefer_valid,
        selection_ranking,
        selection_tournament_variable_k,
        selection_boltzmann
    )
    
    operator_mappings = {
        # Mutation operators
        "mutate_swap_constrained": mutate_swap_constrained,
        "mutate_targeted_player_exchange": mutate_targeted_player_exchange,
        "mutate_shuffle_within_team_constrained": mutate_shuffle_within_team_constrained,
        
        # Crossover operators
        "crossover_one_point_prefer_valid": crossover_one_point_prefer_valid,
        "crossover_uniform_prefer_valid": crossover_uniform_prefer_valid,
        
        # Selection operators
        "selection_tournament_variable_k": selection_tournament_variable_k,
        "selection_ranking": selection_ranking,
        "selection_boltzmann": selection_boltzmann
    }
    
    # Convert string operators to function references in GA config
    for i, ga_config in enumerate(config["ga_configs"]):
        for op_key in ["mut_op", "cross_op", "sel_op"]:
            if isinstance(ga_config[op_key], str) and ga_config[op_key] in operator_mappings:
                config["ga_configs"][i][op_key] = operator_mappings[ga_config[op_key]]
    
    return config

In [ ]:
#  To override default config, uncomment and modify the following:
#  custom_config = {
#     "num_runs": 5,
#     "sa_configs": [
#         {"name": "SA_Custom", "initial_temp": 3000, "final_temp": 0.1, "alpha": 0.97, "iterations_per_temp": 150}
#     ]
# }
# config = get_experiment_config(custom_config)


config = get_experiment_config()

# Hill Climbing

In [ ]:
def run_hill_climbing(players_data: dict, config: dict = None) -> tuple:
    """
    Run Hill Climbing algorithm with multiple configurations.
    
    Args:
        players_data (dict): Dictionary containing player data
        config (dict): Configuration dictionary from get_experiment_config()
        
    Returns:
        tuple: (best_configurations, best_histories, summary_stats_list)
    """
    # Get configuration or use defaults
    if config is None:
        config = get_experiment_config()
    
    hc_section_start_time = time.time()
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] --- Starting Hill Climbing Algorithm Configurations ({config['num_runs']} runs each) ---")
    
    # Dictionaries to store the best results for each HC configuration
    HC_histories = {}
    HC_best_solutions = {}
    summary_stats_list = []
    
    # Process each Hill Climbing configuration
    for hc_config in config["hc_configs"]:
        config_name = hc_config["name"]
        max_iterations = hc_config["max_iterations"]
        
        timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n  [{timestamp_val}] Running HC Configuration: {config_name}")
        
        # Initialize tracking variables for this configuration
        hc_all_fitness_values_config = []
        hc_all_exec_times_config = []
        best_hc_solution_config_overall = None
        best_hc_fitness_config_overall = float("inf")
        best_hc_history_config_overall = []
        
        # Run this HC configuration multiple times
        for i in range(config["num_runs"]):
            timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
            run_count_val = f"{i+1}/{config['num_runs']}"
            print(f"    [{timestamp_val}] HC Run {run_count_val} for {config_name}...")
            start_time_hc_run = time.time()
            
            # Create initial solution
            initial_hc_solution_run = LeagueHillClimbingSolution(
                players_data, 
                num_teams=config["num_teams"], 
                team_size=config["team_size"], 
                max_budget=config["max_budget"]
            )
            
            # Retry generating valid initial solution if needed
            retry_attempts_hc = 0
            max_retry_hc = 5
            while not initial_hc_solution_run.is_valid() and retry_attempts_hc < max_retry_hc:
                print(f"      [{time.strftime('%Y-%m-%d %H:%M:%S')}] HC Run {i+1}: Initial solution invalid, retrying ({retry_attempts_hc+1})...")
                initial_hc_solution_run = LeagueHillClimbingSolution(
                    players_data, 
                    num_teams=config["num_teams"], 
                    team_size=config["team_size"], 
                    max_budget=config["max_budget"]
                )
                retry_attempts_hc += 1
                
            # Skip run if unable to generate valid initial solution
            if not initial_hc_solution_run.is_valid():
                print(f"    [{time.strftime('%Y-%m-%d %H:%M:%S')}] HC Run {i+1} failed to create a valid initial solution after {max_retry_hc} retries. Skipping run.")
                hc_all_fitness_values_config.append(float('nan'))
                hc_all_exec_times_config.append(float('nan'))
                continue

            # Execute Hill Climbing algorithm with this configuration's parameters
            hc_solution_obj_run, hc_fitness_val_run, hc_history_convergence_run = hill_climbing(
                initial_solution=initial_hc_solution_run, 
                players_data=players_data, 
                max_iterations=max_iterations, 
                verbose=False
            )
            end_time_hc_run = time.time()
            hc_exec_time_run = end_time_hc_run - start_time_hc_run
            
            # Store results if valid solution found
            if hc_solution_obj_run:
                hc_all_fitness_values_config.append(hc_fitness_val_run)
                hc_all_exec_times_config.append(hc_exec_time_run)
                # Update best overall solution if current solution is better
                if hc_fitness_val_run < best_hc_fitness_config_overall:
                    best_hc_fitness_config_overall = hc_fitness_val_run
                    best_hc_solution_config_overall = hc_solution_obj_run
                    best_hc_history_config_overall = hc_history_convergence_run
            else:
                print(f"    [{timestamp_val}] HC Run {run_count_val} for {config_name}... did not return a valid solution.")
                hc_all_fitness_values_config.append(float('nan'))
                hc_all_exec_times_config.append(float('nan'))
        
        # Calculate summary statistics for this HC configuration
        hc_mean_fitness_config = np.nanmean(hc_all_fitness_values_config) if hc_all_fitness_values_config else float("nan")
        hc_std_fitness_config = np.nanstd(hc_all_fitness_values_config) if hc_all_fitness_values_config else float("nan")
        hc_mean_exec_time_config = np.nanmean(hc_all_exec_times_config) if hc_all_exec_times_config else float("nan")
        
        # Print summary statistics for this HC configuration
        print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] HC Configuration {config_name} ({config['num_runs']} runs) processing finished.")
        print(f"    Mean Best Fitness: {hc_mean_fitness_config:.4f}")
        print(f"    Std Dev Best Fitness: {hc_std_fitness_config:.4f}")
        print(f"    Mean Execution Time per run: {hc_mean_exec_time_config:.2f}s")
        
        # Generate and save plot for this HC configuration if valid solution found
        if best_hc_solution_config_overall:
            print(f"    Overall Best HC Fitness for {config_name}: {best_hc_fitness_config_overall:.4f}")
            
            # Add this HC configuration's results to summary for later comparison
            summary_stats = {
                "Algorithm": f"{config_name} (SP)",
                "Mean Fitness": hc_mean_fitness_config,
                "Std Dev Fitness": hc_std_fitness_config,
                "Mean Exec Time (s)": hc_mean_exec_time_config,
                "Overall Best Fitness": best_hc_fitness_config_overall,
                "Max Iterations": max_iterations
            }
            summary_stats_list.append(summary_stats)
            
            # Plot this HC configuration's convergence
            plt.figure(figsize=config.get('figure_size', (10, 6)))
            plt.plot(best_hc_history_config_overall, marker=".", linestyle="-")
            plt.title(f"HC Convergence ({config_name}) - Best of {config['num_runs']} Runs - SP")
            plt.xlabel("Improvement Step")
            plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
            plt.grid(True)
            plt.savefig(os.path.join(config['images_dir'], f"hc_convergence_sp_{config_name}.png"))
            plt.show()
            print(f"    [{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved HC convergence plot to {config['images_dir']}/hc_convergence_sp_{config_name}.png")
            plt.close()
            
            # Store this HC configuration's history for the final comparison plot
            HC_histories[config_name] = best_hc_history_config_overall
            HC_best_solutions[config_name] = best_hc_solution_config_overall
        else:
            print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] HC Configuration {config_name} did not find any valid solution across all runs.")
            
            # Add null results to summary
            summary_stats = {
                "Algorithm": f"{config_name} (SP)",
                "Mean Fitness": hc_mean_fitness_config,
                "Std Dev Fitness": hc_std_fitness_config,
                "Mean Exec Time (s)": hc_mean_exec_time_config,
                "Overall Best Fitness": float('nan'),
                "Max Iterations": max_iterations
            }
            summary_stats_list.append(summary_stats)
    
    # Report overall HC section timing
    hc_section_end_time = time.time()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Hill Climbing section took {hc_section_end_time - hc_section_start_time:.2f} seconds.")
    
    return HC_best_solutions, HC_histories, summary_stats_list

In [ ]:
def compare_hill_climbing_runs(hc_histories: dict, images_dir: str = None, config: dict = None) -> None:
    """
    Compare different Hill Climbing configurations.
    
    Args:
        hc_histories (dict): Dictionary of HC configurations and their convergence histories
        images_dir (str): Directory to save output images
        config (dict): Configuration dictionary from get_experiment_config()
    """
    if config is None:
        config = get_experiment_config()
    
    if images_dir is None:
        images_dir = config["images_dir"]
    
    if not hc_histories:
        print("No Hill Climbing histories to compare")
        return
    
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] Comparing Hill Climbing Configurations...")
    
    # Create comparison plot
    plt.figure(figsize=config.get('figure_size', (12, 7)))
    
    # Plot each HC configuration
    for config_name, history in hc_histories.items():
        plt.plot(history, marker='.', linestyle='-', linewidth=config.get('line_width', 2), label=config_name)
    
    plt.title("Hill Climbing Configuration Comparison")
    plt.xlabel("Improvement Step")
    plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
    plt.grid(True)
    plt.legend(loc="best")
    
    # Save the plot
    plot_filename = os.path.join(images_dir, "hc_configurations_comparison.png")
    plt.savefig(plot_filename)
    plt.show()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved Hill Climbing comparison plot to {plot_filename}")
    plt.close()
    
    # Create a summary table of the best final fitness values
    print("\nHill Climbing Configuration Results:")
    print("-" * 50)
    print(f"{'Configuration':<30} {'Final Fitness':<15}")
    print("-" * 50)
    
    for config_name, history in hc_histories.items():
        if history:
            final_fitness = history[-1]
            print(f"{config_name:<30} {final_fitness:<15.4f}")
    
    print("-" * 50)

# Simulated Annealing

In [ ]:
def run_simulated_annealing(players_data: dict, config: dict = None) -> tuple:
    """
    Run Simulated Annealing algorithm with multiple parameter configurations.
    
    Args:
        players_data (dict): Dictionary containing player data
        config (dict): Configuration dictionary from get_experiment_config()
        
    Returns:
        tuple: (best_configurations, best_histories, summary_stats_list)
    """
    # Get configuration or use defaults
    if config is None:
        config = get_experiment_config()
    
    sa_section_start_time = time.time()
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] --- Starting Simulated Annealing Algorithm Configurations ({config['num_runs']} runs each) ---")
    
    # Dictionary to store the best convergence history for each SA configuration
    SA_histories = {}
    SA_best_solutions = {}
    summary_stats_list = []
    
    # Test each SA configuration
    for sa_config in config["sa_configs"]:
        config_name_val = sa_config["name"]
        timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n  [{timestamp_val}] Running SA Configuration: {config_name_val}")
        
        # Initialize tracking variables for this SA configuration
        sa_all_fitness_values_config = []
        sa_all_exec_times_config = []
        best_sa_solution_config_overall = None
        best_sa_fitness_config_overall = float("inf")
        best_sa_history_config_overall = []
        
        # Run this SA configuration multiple times
        for i in range(config["num_runs"]):
            timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
            run_count_val = f"{i+1}/{config['num_runs']}"
            print(f"    [{timestamp_val}] SA Run {run_count_val} for {config_name_val}...")
            start_time_sa_run = time.time()
            
            # Create initial solution for this SA run
            initial_sa_solution_run = LeagueSASolution(
                players_data, 
                num_teams=config["num_teams"], 
                team_size=config["team_size"], 
                max_budget=config["max_budget"]
            )
            
            # Retry generating a valid initial solution if needed
            retry_attempts_sa = 0
            max_retry_sa = 5
            while not initial_sa_solution_run.is_valid() and retry_attempts_sa < max_retry_sa:
                print(f"      [{time.strftime('%Y-%m-%d %H:%M:%S')}] SA Run {i+1}: Initial solution invalid, retrying ({retry_attempts_sa+1})...")
                initial_sa_solution_run = LeagueSASolution(
                    players_data, 
                    num_teams=config["num_teams"], 
                    team_size=config["team_size"], 
                    max_budget=config["max_budget"]
                )
                retry_attempts_sa += 1
            
            # Skip run if unable to generate valid initial solution
            if not initial_sa_solution_run.is_valid():
                print(f"    [{time.strftime('%Y-%m-%d %H:%M:%S')}] SA Run {i+1} for {config_name_val} failed to create a valid initial solution after {max_retry_sa} retries. Skipping run.")
                sa_all_fitness_values_config.append(float('nan'))
                sa_all_exec_times_config.append(float('nan'))
                continue
            
            # Execute Simulated Annealing with this configuration's parameters
            sa_solution_obj_run, sa_fitness_val_run, sa_history_convergence_run = simulated_annealing(
                initial_solution=initial_sa_solution_run,
                players_data=players_data,
                initial_temp=sa_config["initial_temp"],
                final_temp=sa_config["final_temp"],
                alpha=sa_config["alpha"],
                iterations_per_temp=sa_config["iterations_per_temp"],
                verbose=False
            )
            end_time_sa_run = time.time()
            sa_exec_time_run = end_time_sa_run - start_time_sa_run
            
            # Process results if valid solution found
            if sa_solution_obj_run:
                sa_all_fitness_values_config.append(sa_fitness_val_run)
                sa_all_exec_times_config.append(sa_exec_time_run)
                
                # Update best solution if current solution is better
                if sa_fitness_val_run < best_sa_fitness_config_overall:
                    best_sa_fitness_config_overall = sa_fitness_val_run
                    best_sa_solution_config_overall = sa_solution_obj_run
                    best_sa_history_config_overall = sa_history_convergence_run
            else:
                print(f"    [{timestamp_val}] SA Run {run_count_val} for {config_name_val}... did not return a valid solution object.")
                sa_all_fitness_values_config.append(float('nan'))
                sa_all_exec_times_config.append(float('nan'))
        
        # Calculate summary statistics for this SA configuration
        sa_mean_fitness_config = np.nanmean(sa_all_fitness_values_config) if sa_all_fitness_values_config else float("nan")
        sa_std_fitness_config = np.nanstd(sa_all_fitness_values_config) if sa_all_fitness_values_config else float("nan")
        sa_mean_exec_time_config = np.nanmean(sa_all_exec_times_config) if sa_all_exec_times_config else float("nan")
        
        # Print summary statistics for this SA configuration
        print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] SA Configuration {config_name_val} ({config['num_runs']} runs) processing finished.")
        print(f"    Mean Best Fitness: {sa_mean_fitness_config:.4f}")
        print(f"    Std Dev Best Fitness: {sa_std_fitness_config:.4f}")
        print(f"    Mean Execution Time per run: {sa_mean_exec_time_config:.2f}s")
        
        # Generate and save plot for this SA configuration if valid solution found
        if best_sa_solution_config_overall:
            print(f"    Overall Best SA Fitness for {config_name_val}: {best_sa_fitness_config_overall:.4f}")
            
            # Add this SA configuration's results to summary for later comparison
            summary_stats = {
                "Algorithm": f"{config_name_val} (SP)",
                "Mean Fitness": sa_mean_fitness_config,
                "Std Dev Fitness": sa_std_fitness_config,
                "Mean Exec Time (s)": sa_mean_exec_time_config,
                "Overall Best Fitness": best_sa_fitness_config_overall,
                "Initial Temp": sa_config["initial_temp"],
                "Final Temp": sa_config["final_temp"],
                "Alpha": sa_config["alpha"],
                "Iterations/Temp": sa_config["iterations_per_temp"]
            }
            summary_stats_list.append(summary_stats)
            
            # Plot this SA configuration's convergence
            plt.figure(figsize=config.get('figure_size', (10, 6)))
            plt.plot(best_sa_history_config_overall, marker=".", linestyle="-")
            plt.title(f"SA Convergence ({config_name_val}) - Best of {config['num_runs']} Runs - SP")
            plt.xlabel("Iteration")
            plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
            plt.grid(True)
            plt.savefig(os.path.join(config['images_dir'], f"sa_convergence_sp_{config_name_val}.png"))
            plt.show()
            print(f"    [{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved SA convergence plot to {config['images_dir']}/sa_convergence_sp_{config_name_val}.png")
            plt.close()
            
            # Store this SA configuration's history for the final comparison plot
            SA_histories[config_name_val] = best_sa_history_config_overall
            SA_best_solutions[config_name_val] = best_sa_solution_config_overall
        else:
            print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] SA Configuration {config_name_val} did not find any valid solution across all runs that produced a best overall.")
            
            # Add null results to summary
            summary_stats = {
                "Algorithm": f"{config_name_val} (SP)",
                "Mean Fitness": sa_mean_fitness_config,
                "Std Dev Fitness": sa_std_fitness_config,
                "Mean Exec Time (s)": sa_mean_exec_time_config,
                "Overall Best Fitness": float('nan'),
                "Initial Temp": sa_config["initial_temp"],
                "Final Temp": sa_config["final_temp"],
                "Alpha": sa_config["alpha"],
                "Iterations/Temp": sa_config["iterations_per_temp"]
            }
            summary_stats_list.append(summary_stats)
    
    # Report overall SA section timing
    sa_section_end_time = time.time()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Simulated Annealing section took {sa_section_end_time - sa_section_start_time:.2f} seconds.")
    
    return SA_best_solutions, SA_histories, summary_stats_list

In [ ]:
def compare_sa_configurations(sa_histories: dict, images_dir: str = None, config: dict = None) -> None:
    """
    Compare different Simulated Annealing configurations.
    
    Args:
        sa_histories (dict): Dictionary of SA configurations and their convergence histories
        images_dir (str): Directory to save output images
        config (dict): Configuration dictionary from get_experiment_config()
    """
    if config is None:
        config = get_experiment_config()
    
    if images_dir is None:
        images_dir = config["images_dir"]
    
    if not sa_histories:
        print("No Simulated Annealing histories to compare")
        return
    
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] Comparing Simulated Annealing Configurations...")
    
    # Create comparison plot
    plt.figure(figsize=config.get('figure_size', (12, 7)))
    
    # Plot each SA configuration
    for config_name, history in sa_histories.items():
        plt.plot(history, marker='.', linestyle='-', linewidth=config.get('line_width', 2), label=config_name)
    
    plt.title("Simulated Annealing Configuration Comparison")
    plt.xlabel("Iteration")
    plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
    plt.grid(True)
    plt.legend(loc="best")
    
    # Save the plot
    plot_filename = os.path.join(images_dir, "sa_configurations_comparison.png")
    plt.savefig(plot_filename)
    plt.show()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved Simulated Annealing comparison plot to {plot_filename}")
    plt.close()
    
    # Create a summary table of the best final fitness values
    print("\nSimulated Annealing Configuration Results:")
    print("-" * 50)
    print(f"{'Configuration':<30} {'Final Fitness':<15}")
    print("-" * 50)
    
    for config_name, history in sa_histories.items():
        if history:
            final_fitness = min(history)  # Get the best fitness value
            print(f"{config_name:<30} {final_fitness:<15.4f}")
    
    print("-" * 50)

# Genetic Algorithms

In [ ]:
def run_genetic_algorithm(players_data: dict, config: dict = None) -> tuple:
    """
    Run Genetic Algorithm with multiple configurations.
    
    Args:
        players_data (dict): Dictionary containing player data
        config (dict): Configuration dictionary from get_experiment_config()
        
    Returns:
        tuple: (best_configurations, best_histories, summary_stats_list)
    """
    # Get configuration or use defaults
    if config is None:
        config = get_experiment_config()
    
    ga_section_start_time = time.time()
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] --- Starting Genetic Algorithm Configurations ({config['num_runs']} runs each) ---")
    
    # Dictionaries to store the best results for each GA configuration
    GA_histories = {}
    GA_best_solutions = {}
    summary_stats_list = []
    
    # Test each GA configuration
    for ga_config in config["ga_configs"]:
        config_name_val = ga_config["name"]
        timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n  [{timestamp_val}] Running GA Configuration: {config_name_val}")
        
        # Initialize tracking variables for this GA configuration
        ga_all_fitness_values_config = []
        ga_all_exec_times_config = []
        best_ga_solution_config_overall = None
        best_ga_fitness_config_overall = float("inf")
        best_ga_history_config_overall = []
        
        # Run this GA configuration multiple times
        for i in range(config["num_runs"]):
            timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
            run_count_val = f"{i+1}/{config['num_runs']}"
            print(f"    [{timestamp_val}] GA Run {run_count_val} for {config_name_val}...")
            start_time_ga_run = time.time()
            
            # Extract specific GA configuration parameters
            pop_size = ga_config["pop_size"]
            generations = ga_config["gens"]
            mutation_rate = ga_config["mut_rate"]
            elite_size = ga_config["elite"]
            mutation_operator = ga_config["mut_op"]
            crossover_operator = ga_config["cross_op"]
            selection_operator = ga_config["sel_op"]
            tournament_k = ga_config.get("tourn_k")
            boltzmann_temp = ga_config.get("boltz_temp")
            
            # Execute the Genetic Algorithm with this configuration
            try:
                # Create selection operator kwargs if needed
                selection_kwargs = {}
                if selection_operator == selection_tournament_variable_k and tournament_k is not None:
                    selection_kwargs["tournament_k"] = tournament_k
                if selection_operator == selection_boltzmann and boltzmann_temp is not None:
                    selection_kwargs["temperature"] = boltzmann_temp
                
                # Run GA algorithm with this configuration
                ga_best_solution, ga_best_fitness, ga_history = genetic_algorithm(
                    players_data=players_data,
                    num_teams=config["num_teams"],
                    team_size=config["team_size"],
                    max_budget=config["max_budget"],
                    population_size=pop_size,
                    generations=generations,
                    mutation_rate=mutation_rate,
                    elite_size=elite_size,
                    mutation_operator=mutation_operator,
                    crossover_operator=crossover_operator,
                    selection_operator=selection_operator,
                    selection_kwargs=selection_kwargs,
                    verbose=False
                )
                
                end_time_ga_run = time.time()
                ga_exec_time_run = end_time_ga_run - start_time_ga_run
                
                # Process results if valid solution found
                if ga_best_solution is not None:
                    ga_all_fitness_values_config.append(ga_best_fitness)
                    ga_all_exec_times_config.append(ga_exec_time_run)
                    
                    # Update best solution if current solution is better
                    if ga_best_fitness < best_ga_fitness_config_overall:
                        best_ga_fitness_config_overall = ga_best_fitness
                        best_ga_solution_config_overall = ga_best_solution
                        best_ga_history_config_overall = ga_history
                else:
                    print(f"    [{timestamp_val}] GA Run {run_count_val} for {config_name_val}... did not return a valid solution.")
                    ga_all_fitness_values_config.append(float('nan'))
                    ga_all_exec_times_config.append(float('nan'))
                    
            except Exception as e:
                timestamp_val = time.strftime("%Y-%m-%d %H:%M:%S")
                print(f"    [{timestamp_val}] Error in GA Run {run_count_val} for {config_name_val}: {str(e)}")
                ga_all_fitness_values_config.append(float('nan'))
                ga_all_exec_times_config.append(float('nan'))
                continue
        
        # Calculate summary statistics for this GA configuration
        ga_mean_fitness_config = np.nanmean(ga_all_fitness_values_config) if ga_all_fitness_values_config else float("nan")
        ga_std_fitness_config = np.nanstd(ga_all_fitness_values_config) if ga_all_fitness_values_config else float("nan")
        ga_mean_exec_time_config = np.nanmean(ga_all_exec_times_config) if ga_all_exec_times_config else float("nan")
        
        # Print summary statistics for this GA configuration
        print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] GA Configuration {config_name_val} ({config['num_runs']} runs) processing finished.")
        print(f"    Mean Best Fitness: {ga_mean_fitness_config:.4f}")
        print(f"    Std Dev Best Fitness: {ga_std_fitness_config:.4f}")
        print(f"    Mean Execution Time per run: {ga_mean_exec_time_config:.2f}s")
        
        # Get readable operator names for reporting
        mutation_op_name = mutation_operator.__name__ if hasattr(mutation_operator, "__name__") else str(mutation_operator)
        crossover_op_name = crossover_operator.__name__ if hasattr(crossover_operator, "__name__") else str(crossover_operator)
        selection_op_name = selection_operator.__name__ if hasattr(selection_operator, "__name__") else str(selection_operator)
        
        # Generate and save plot for this GA configuration if valid solution found
        if best_ga_solution_config_overall is not None:
            print(f"    Overall Best GA Fitness for {config_name_val}: {best_ga_fitness_config_overall:.4f}")
            
            # Add this GA configuration's results to summary for later comparison
            summary_stats = {
                "Algorithm": f"{config_name_val} (SP)",
                "Mean Fitness": ga_mean_fitness_config,
                "Std Dev Fitness": ga_std_fitness_config,
                "Mean Exec Time (s)": ga_mean_exec_time_config,
                "Overall Best Fitness": best_ga_fitness_config_overall,
                "Population Size": pop_size,
                "Generations": generations,
                "Mutation Rate": mutation_rate,
                "Elite Size": elite_size,
                "Mutation Op": mutation_op_name,
                "Crossover Op": crossover_op_name,
                "Selection Op": selection_op_name
            }
            summary_stats_list.append(summary_stats)
            
            # Plot this GA configuration's convergence
            plt.figure(figsize=config.get('figure_size', (10, 6)))
            plt.plot(best_ga_history_config_overall, marker=".", linestyle="-")
            plt.title(f"GA Convergence ({config_name_val}) - Best of {config['num_runs']} Runs - SP")
            plt.xlabel("Generation")
            plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
            plt.grid(True)
            plt.savefig(os.path.join(config['images_dir'], f"ga_convergence_sp_{config_name_val}.png"))
            plt.show()
            print(f"    [{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved GA convergence plot to {config['images_dir']}/ga_convergence_sp_{config_name_val}.png")
            plt.close()
            
            # Store this GA configuration's history for the final comparison plot
            GA_histories[config_name_val] = best_ga_history_config_overall
            GA_best_solutions[config_name_val] = best_ga_solution_config_overall
        else:
            print(f"  [{time.strftime('%Y-%m-%d %H:%M:%S')}] GA Configuration {config_name_val} did not find any valid solution across all runs.")
            
            # Add null results to summary
            summary_stats = {
                "Algorithm": f"{config_name_val} (SP)",
                "Mean Fitness": ga_mean_fitness_config,
                "Std Dev Fitness": ga_std_fitness_config,
                "Mean Exec Time (s)": ga_mean_exec_time_config,
                "Overall Best Fitness": float('nan'),
                "Population Size": pop_size,
                "Generations": generations,
                "Mutation Rate": mutation_rate,
                "Elite Size": elite_size,
                "Mutation Op": mutation_op_name,
                "Crossover Op": crossover_op_name,
                "Selection Op": selection_op_name
            }
            summary_stats_list.append(summary_stats)
    
    # Report overall GA section timing
    ga_section_end_time = time.time()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Genetic Algorithm section took {ga_section_end_time - ga_section_start_time:.2f} seconds.")
    
    return GA_best_solutions, GA_histories, summary_stats_list

In [ ]:
def compare_ga_configurations(ga_histories: dict, images_dir: str = None, config: dict = None) -> None:
    """
    Compare different Genetic Algorithm configurations.
    
    Args:
        ga_histories (dict): Dictionary of GA configurations and their convergence histories
        images_dir (str): Directory to save output images
        config (dict): Configuration dictionary from get_experiment_config()
    """
    if config is None:
        config = get_experiment_config()
    
    if images_dir is None:
        images_dir = config["images_dir"]
    
    if not ga_histories:
        print("No Genetic Algorithm histories to compare")
        return
    
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] Comparing Genetic Algorithm Configurations...")
    
    # Create comparison plot
    plt.figure(figsize=config.get('figure_size', (12, 7)))
    
    # Plot each GA configuration
    for config_name, history in ga_histories.items():
        plt.plot(history, marker='.', linestyle='-', linewidth=config.get('line_width', 2), label=config_name)
    
    plt.title("Genetic Algorithm Configuration Comparison")
    plt.xlabel("Generation")
    plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
    plt.grid(True)
    plt.legend(loc="best")
    
    # Save the plot
    plot_filename = os.path.join(images_dir, "ga_configurations_comparison.png")
    plt.savefig(plot_filename)
    plt.show()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved Genetic Algorithm comparison plot to {plot_filename}")
    plt.close()
    
    # Create a summary table of the best final fitness values
    print("\nGenetic Algorithm Configuration Results:")
    print("-" * 80)
    print(f"{'Configuration':<50} {'Final Fitness':<15}")
    print("-" * 80)
    
    for config_name, history in ga_histories.items():
        if history:
            final_fitness = history[-1]  # Last generation fitness
            print(f"{config_name:<50} {final_fitness:<15.4f}")
    
    print("-" * 80)

# Global Comparison


def compare_algorithms(
    hc_histories: dict = None, 
    sa_histories: dict = None, 
    ga_histories: dict = None, 
    selected_configs: dict = None,
    images_dir: str = None, 
    config: dict = None
) -> None:
    """
    Compare selected configurations from each algorithm type.
    
    Args:
        hc_histories (dict): Dictionary of HC configurations and their convergence histories
        sa_histories (dict): Dictionary of SA configurations and their convergence histories
        ga_histories (dict): Dictionary of GA configurations and their convergence histories
        selected_configs (dict): Custom override for which configs to compare in format:
                               {'hc': ['HC_Config1', 'HC_Config2'], 
                                'sa': ['SA_Config1'], 
                                'ga': ['GA_Config3', 'GA_Config4']}
                               If None, best config from each algorithm type is selected.
        images_dir (str): Directory to save output images
        config (dict): Configuration dictionary from get_experiment_config()
    """
    if config is None:
        config = get_experiment_config()
    
    if images_dir is None:
        images_dir = config["images_dir"]
    
    # Initialize empty dictionaries if not provided
    hc_histories = hc_histories or {}
    sa_histories = sa_histories or {}
    ga_histories = ga_histories or {}
    selected_configs = selected_configs or {}
    
    if not hc_histories and not sa_histories and not ga_histories:
        print("No algorithm histories to compare")
        return
    
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] Comparing Selected Algorithm Configurations...")
    
    configs_to_plot = {}
    
    # HILL CLIMBING: Get configs to plot
    if selected_configs.get('hc'):
        # Use explicitly selected HC configs
        for hc_name in selected_configs['hc']:
            if hc_name in hc_histories:
                configs_to_plot[f"HC: {hc_name}"] = {
                    'history': hc_histories[hc_name],
                    'final_fitness': hc_histories[hc_name][-1] if hc_histories[hc_name] else float('inf')
                }
            else:
                print(f"Warning: Selected HC config '{hc_name}' not found in histories")
    elif hc_histories:
        # Find best HC config
        best_hc_config = None
        best_hc_history = None
        best_hc_fitness = float('inf')
        
        for config_name, history in hc_histories.items():
            if history and history[-1] < best_hc_fitness:
                best_hc_config = config_name
                best_hc_history = history
                best_hc_fitness = history[-1]
        
        if best_hc_config:
            configs_to_plot[f"HC: {best_hc_config}"] = {
                'history': best_hc_history,
                'final_fitness': best_hc_fitness
            }
    
    # SIMULATED ANNEALING: Get configs to plot
    if selected_configs.get('sa'):
        # Use explicitly selected SA configs
        for sa_name in selected_configs['sa']:
            if sa_name in sa_histories:
                min_fitness = min(sa_histories[sa_name]) if sa_histories[sa_name] else float('inf')
                configs_to_plot[f"SA: {sa_name}"] = {
                    'history': sa_histories[sa_name],
                    'final_fitness': min_fitness
                }
            else:
                print(f"Warning: Selected SA config '{sa_name}' not found in histories")
    elif sa_histories:
        # Find best SA config
        best_sa_config = None
        best_sa_history = None
        best_sa_fitness = float('inf')
        
        for config_name, history in sa_histories.items():
            if history and min(history) < best_sa_fitness:
                best_sa_config = config_name
                best_sa_history = history
                best_sa_fitness = min(history)
        
        if best_sa_config:
            configs_to_plot[f"SA: {best_sa_config}"] = {
                'history': best_sa_history,
                'final_fitness': best_sa_fitness
            }
    
    # GENETIC ALGORITHM: Get configs to plot
    if selected_configs.get('ga'):
        # Use explicitly selected GA configs
        for ga_name in selected_configs['ga']:
            if ga_name in ga_histories:
                configs_to_plot[f"GA: {ga_name}"] = {
                    'history': ga_histories[ga_name],
                    'final_fitness': ga_histories[ga_name][-1] if ga_histories[ga_name] else float('inf')
                }
            else:
                print(f"Warning: Selected GA config '{ga_name}' not found in histories")
    elif ga_histories:
        # Find best GA config
        best_ga_config = None
        best_ga_history = None
        best_ga_fitness = float('inf')
        
        for config_name, history in ga_histories.items():
            if history and history[-1] < best_ga_fitness:
                best_ga_config = config_name
                best_ga_history = history
                best_ga_fitness = history[-1]
        
        if best_ga_config:
            configs_to_plot[f"GA: {best_ga_config}"] = {
                'history': best_ga_history,
                'final_fitness': best_ga_fitness
            }
    
    if not configs_to_plot:
        print("No valid configurations to compare")
        return
    
    # Create comparison plot
    plt.figure(figsize=config.get('figure_size', (14, 8)))
    
    # Custom colors for each algorithm family
    hc_colors = ['#1f77b4', '#aec7e8', '#3182bd', '#80b1d3']  # Blues
    sa_colors = ['#ff7f0e', '#ffbb78', '#e6550d', '#fdae6b']  # Oranges
    ga_colors = ['#2ca02c', '#98df8a', '#a1d99b', '#b2e2e2']  # Greens
    
    hc_count, sa_count, ga_count = 0, 0, 0
    
    # Plot each selected configuration
    for config_name, config_data in configs_to_plot.items():
        history = config_data['history']
        final_fitness = config_data['final_fitness']
        
        if not history:
            continue
            
        # Normalize x-axis to percentage of completion for fair comparison
        x_norm = np.linspace(0, 100, len(history))
        
        # Select color based on algorithm type
        if config_name.startswith("HC:"):
            color = hc_colors[hc_count % len(hc_colors)]
            hc_count += 1
        elif config_name.startswith("SA:"):
            color = sa_colors[sa_count % len(sa_colors)]
            sa_count += 1
        elif config_name.startswith("GA:"):
            color = ga_colors[ga_count % len(ga_colors)]
            ga_count += 1
        else:
            color = None  # Use default color cycle
        
        plt.plot(x_norm, history, marker='.', linestyle='-', color=color,
                 linewidth=config.get('line_width', 2), 
                 label=f"{config_name} (Final: {final_fitness:.4f})")
    
    plt.title("Algorithm Configuration Comparison")
    plt.xlabel("Percentage of Algorithm Completion")
    plt.ylabel("Fitness (Std Dev of Avg Team Skills)")
    plt.grid(True)
    plt.legend(loc="best")
    
    # Save the plot
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    plot_filename = os.path.join(images_dir, f"algorithm_comparison_{timestamp}.png")
    plt.savefig(plot_filename)
    plt.show()
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Saved Algorithm comparison plot to {plot_filename}")
    plt.close()
    
    # Create a summary table
    print("\nSelected Algorithm Configuration Results:")
    print("-" * 100)
    print(f"{'Algorithm':<50} {'Best Fitness':<15}")
    print("-" * 100)
    
    # Sort by fitness for the table display
    sorted_configs = sorted(configs_to_plot.items(), key=lambda x: x[1]['final_fitness'])
    
    for config_name, config_data in sorted_configs:
        print(f"{config_name:<50} {config_data['final_fitness']:<15.4f}")
    
    print("-" * 100)
    
    # Identify overall best
    if sorted_configs:
        best_config_name, best_config_data = sorted_configs[0]
        print(f"\nBest Configuration: {best_config_name}")
        print(f"Best Fitness Achieved: {best_config_data['final_fitness']:.4f}")

In [ ]:
compare_all_algorithms(hc_histories, sa_histories, ga_histories, config=config)

# Guide: Using the Algorithm Comparison Functions

This guide explains how to use the algorithm comparison functions in our optimization framework, focusing particularly on the flexible `compare_all_algorithms` function.

## Table of Contents

1. [Basic Comparison](#basic-comparison)
2. [Custom Configuration Selection](#custom-configuration-selection)
3. [Algorithm-Specific Comparisons](#algorithm-specific-comparisons)
4. [Advanced Use Cases](#advanced-use-cases)
5. [Output and Interpretation](#output-and-interpretation)

## Basic Comparison

The simplest way to compare algorithms is to let the function automatically select the best configuration from each algorithm type:

```python
# Assuming you have already run your algorithms and collected histories
compare_all_algorithms(
    hc_histories=hc_results[1], 
    sa_histories=sa_results[1], 
    ga_histories=ga_results[1],
    config=config
)
```

This will:

- Find the best Hill Climbing configuration based on final fitness
- Find the best Simulated Annealing configuration based on minimum fitness
- Find the best Genetic Algorithm configuration based on final generation fitness
- Plot all three on a normalized scale (0-100% of algorithm completion)
- Print a summary table of results

### Custom Configuration Selection
To compare specific algorithm configurations instead of just the best ones:

```python
# Define which specific configurations to include
selected_configs = {
    'hc': ['HC_Default', 'HC_LongRun'],
    'sa': ['SA_Default', 'SA_MoreExploration'],
    'ga': ['GA_Config_1_SwapConst1PtPreferVTournVarK', 'GA_Config_3_ShuffleTeamConst1PtPreferVBoltz']
}

# Run the comparison with your selection
compare_all_algorithms(
    hc_histories=hc_results[1], 
    sa_histories=sa_results[1], 
    ga_histories=ga_results[1],
    selected_configs=selected_configs,
    config=config
)
```

This gives you complete control over which specific configurations to include in the comparison.

### Algorithm-Specific Comparisons

If you only want to compare certain algorithm types:

# Only compare Simulated Annealing and Genetic Algorithm (no Hill Climbing)
compare_all_algorithms(
    sa_histories=sa_results[1], 
    ga_histories=ga_results[1],
    config=config
)

# Only compare different Hill Climbing configurations
selected_configs = {
    'hc': ['HC_Default', 'HC_LongRun', 'HC_ShortRun']
}

compare_all_algorithms(
    hc_histories=hc_results[1],
    selected_configs=selected_configs,
    config=config
)
```

### Advanced Use Cases

#### Comparing Against a Baseline   
To compare new configurations against a known baseline:

# Define baseline and experimental configurations
selected_configs = {
    'hc': ['HC_Baseline'],
    'sa': ['SA_Baseline', 'SA_Experimental1', 'SA_Experimental2'],
    'ga': ['GA_Baseline']
}

compare_all_algorithms(
    hc_histories=hc_results[1], 
    sa_histories=sa_results[1], 
    ga_histories=ga_results[1],
    selected_configs=selected_configs,
    config=config
)

#### Time-Constrained Comparison
If you want to evaluate algorithm performance within a specific time budget:

# First, filter your histories based on execution time
time_constrained_histories = {
    'hc': {k: v for k, v in hc_results[1].items() if execution_times['hc'][k] < max_time},
    'sa': {k: v for k, v in sa_results[1].items() if execution_times['sa'][k] < max_time},
    'ga': {k: v for k, v in ga_results[1].items() if execution_times['ga'][k] < max_time}
}

# Then run the comparison
compare_all_algorithms(
    hc_histories=time_constrained_histories['hc'],
    sa_histories=time_constrained_histories['sa'],
    ga_histories=time_constrained_histories['ga'],
    config=config
)

### Output and Interpretation
The function produces several outputs:

Visual plot with normalized fitness curves, saved to the images directory
Console summary table showing each configuration and its best fitness
Overall best configuration identified with its final fitness value

#### Understanding the Plot
X-axis: Normalized algorithm progress (0-100%)
Y-axis: Fitness value (lower is better)
Color coding:
Hill Climbing configurations: Blue family
Simulated Annealing configurations: Orange family
Genetic Algorithm configurations: Green family

#### Interpreting Results
When analyzing the results, consider:

Final fitness value: The quality of the final solution
Convergence speed: How quickly the algorithm approaches good solutions
Stability: Consistent improvement vs. erratic behavior
Exploration vs. exploitation: Initial exploration phase followed by refinemen